In [0]:
  USE CATALOG tibia_analytics;
   USE SCHEMA bronze;
SET TIME ZONE 'UTC';

In [0]:
COPY INTO raw_highscores
FROM (
  SELECT SHA1(
           CONCAT_WS(
             '|',
             _METADATA.FILE_PATH,
             CAST(
               _METADATA.FILE_MODIFICATION_TIME AS STRING
             )
           )
         )                                AS raw_highscores_id,
         FROM_JSON(
           TO_JSON(highscores),
           'STRUCT<
             world: STRING,
             category: STRING,
             vocation: STRING,
             highscore_age: INT,
             highscore_list: ARRAY<STRUCT<
               rank: INT,
               name: STRING,
               title: STRING,
               vocation: STRING,
               world: STRING,
               level: INT,
               value: INT
             >>,
             highscore_page: STRUCT<
               current_page: INT,
               total_pages: INT,
               total_records: INT
             >
           >'
         )                                AS highscores,
         FROM_JSON(
           TO_JSON(information),
           'STRUCT<
             api: STRUCT<
               version: INT,
               release: STRING,
               commit: STRING
             >,
             timestamp: STRING,
             tibia_urls: ARRAY<STRING>,
             status: STRUCT<
               error: INT,
               http_code: INT,
               message: STRING
             >
           >'
         )                                AS information,
         CAST(
           REGEXP_EXTRACT(
             _METADATA.FILE_PATH,
             '([0-9]{4}-[0-9]{2}-[0-9]{2})',
             1
           ) AS DATE
         )                                AS source_file_date,
         _METADATA.FILE_PATH              AS source_file_path,
         _METADATA.FILE_MODIFICATION_TIME AS source_file_modified_at,
         CURRENT_TIMESTAMP()              AS ingested_at
    FROM '/Volumes/tibia_analytics/bronze/raw/highscores/*'
)
    FILEFORMAT = JSON
FORMAT_OPTIONS ('multiLine'   = 'true')
  COPY_OPTIONS ('mergeSchema' = 'false');